In [1]:
import pandas as pd, numpy as np, os
print(os.getcwd())      # should end in ...\nifty50-direction-forecasting
d = pd.read_csv("nifty50_features.csv", parse_dates=["Date"])
print(d.shape, d["Date"].min().date(), d["Date"].max().date())
print(d["crude_chg"].min(), d["crude_chg"].max())

C:\Users\Kaif\projects\nifty50-direction-forecasting
(4071, 26) 2010-03-17 2026-10-12
-305.9660652724544 39.53717822647376


In [3]:
bad = (d["crude"] <= 0) | (d["crude"].shift() <= 0)   
d.loc[bad, "crude_chg"] = 0
d["crude_chg"] = d["crude_chg"].clip(-20, 20)

print(d["crude_chg"].min(), d["crude_chg"].max())   
d.to_csv("nifty50_features.csv", index=False)

-20.0 20.0


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

features = ["ret", "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5",
            "ma10_ratio", "ma50_ratio", "vol_21d", "rsi14",
            "vix", "vix_chg", "usdinr_chg", "sp500_chg", "crude_chg", "dow"]

train = d[d["Date"] < "2020-01-01"]
val   = d[(d["Date"] >= "2020-01-01") & (d["Date"] < "2023-01-01")]
test  = d[d["Date"] >= "2023-01-01"]      # leave untouched until the very end

print(len(train), len(val), len(test))
y_val = val["target_up"]
print("Always up:     ", round(y_val.mean(), 3))
print("Same as today: ", round(accuracy_score(y_val, (val["ret"] > 0).astype(int)), 3))

2394 746 931
Always up:      0.548
Same as today:  0.568


In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=50, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=42),
}

rows = []
for name, m in models.items():
    m.fit(train[features], train["target_up"])
    pred = m.predict(val[features])
    proba = m.predict_proba(val[features])[:, 1]
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_val, pred),
                 "precision": precision_score(y_val, pred, zero_division=0),
                 "recall": recall_score(y_val, pred),
                 "roc_auc": roc_auc_score(y_val, proba)})
print(pd.DataFrame(rows).round(3))